In [ ]:
%use kandy

In [ ]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardOpenOption

val kotlincNative = System.getProperty("user.home") + "/.konan/kotlin-native-prebuilt-linux-x86_64-2.2.10/bin/kotlinc-native"
val tempDir = Files.createTempDirectory("knative-perf")

fun runProcess(vararg command: String): String =
    ProcessBuilder(command.toList())
        .directory(tempDir.toFile())
        .start()
        .also {
            if (it.waitFor() != 0) throw Exception("Process failed with code ${it.exitValue()}")
        }
        .inputStream.bufferedReader().readText()

val source = """
import kotlin.concurrent.Volatile
import kotlin.time.measureTime
import kotlin.time.DurationUnit

@Volatile
private var blackhole: Any? = null
@Volatile
private var string = "abc".repeat(100000)

fun main() {
    repeat(1_000_000) { blackhole = string.substring(1000, 5000) }
    repeat(10_000) {
        measureTime {
            repeat(100) {
                blackhole = string.substring(1000, 5000)
            }
        }.let { println(it.toDouble(DurationUnit.SECONDS)) }
    }
}
"""

val sourceFile = tempDir.resolve("benchmark.kt")
val outputBinary = tempDir.resolve("benchmark.kexe")

Files.writeString(
    sourceFile,
    source,
    StandardOpenOption.CREATE,
)

runProcess(
        kotlincNative,
        sourceFile.toString(),
        "-o", outputBinary.toString()
)

val times = runProcess(outputBinary.toString())
    .split("\n")
    .mapNotNull { it.takeIf { it.isNotEmpty() }?.toDouble() }
    .toList()

In [ ]:
val threshold = 0.00005

In [ ]:
plot {
    histogram(times, binsOption = BinsOption.byNumber(100))
    vLine {
        xIntercept.constant(threshold)
    }
}

In [ ]:
val (noGC, GC) = times.partition { it < threshold }
val avgNoGC = noGC.average()
GC.sumOf { it - avgNoGC } / times.sum()